# Draw and Predict
Draw a character in the canvas, then click **Predict**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import ipywidgets as widgets
from ipycanvas import Canvas, hold_canvas
from IPython.display import display, clear_output
from PIL import Image, ImageOps
import numpy as np
import matplotlib.pyplot as plt
import string, os
from scipy import ndimage
from scipy.signal import find_peaks

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classes     = list(string.digits + string.ascii_uppercase + string.ascii_lowercase)
NUM_CLASSES = 62
CONF_THRESHOLD = 0.35   # predictions below this are shown as '?'
print(f"Device: {device}")

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.shortcut = (
            nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            ) if stride != 1 or in_ch != out_ch else nn.Identity()
        )
    def forward(self, x):
        return F.relu(self.body(x) + self.shortcut(x))


class HandwritingResNet(nn.Module):
    def __init__(self, num_classes=62):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True)
        )
        self.layer1 = nn.Sequential(ResidualBlock(32, 64, stride=2),  ResidualBlock(64, 64))
        self.layer2 = nn.Sequential(ResidualBlock(64, 128, stride=2), ResidualBlock(128, 128))
        self.layer3 = nn.Sequential(ResidualBlock(128, 256, stride=2), ResidualBlock(256, 256))
        self.pool   = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(256, num_classes))
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.classifier(x)


CKPT = './checkpoints/best_model.pth'
model = HandwritingResNet(NUM_CLASSES).to(device)
if os.path.exists(CKPT):
    model.load_state_dict(torch.load(CKPT, map_location=device))
    model.eval()
    print(f"Model loaded -> {CKPT}")
else:
    print(f"Checkpoint not found at {CKPT}. Train the model first.")


In [ ]:
def split_by_projection(crop, x_offset, ref_height):
    """
    Two-phase splitting of a wide blob into individual characters.

    Phase 1 — Gap splitting:
        Find columns where ink density is near-zero.  These are true empty
        spaces between characters.  This safely splits well-separated letters
        WITHOUT cutting inside characters like 'v', 'w', 'm' (which have no
        zero columns — every column touches some part of the stroke).

    Phase 2 — Valley splitting (fallback for touching / cursive letters):
        If no true gaps exist (letters are physically touching), look for
        deep valleys (prominence ≥ 40 % of max ink).  The high threshold
        avoids splitting within single characters.
    """
    h, w = crop.shape
    est_char_w = max(10, ref_height * 0.65)   # one char ≈ 65 % of its height wide

    if w < est_char_w * 1.6:                  # too narrow for multiple chars
        return [(crop, x_offset)]

    col_sum  = (crop > 30).astype(float).sum(axis=0)
    smooth_w = max(3, int(est_char_w * 0.10))
    smoothed = ndimage.uniform_filter1d(col_sum, size=smooth_w)
    max_ink  = smoothed.max()

    # ── Phase 1: gap-based (near-zero columns) ────────────────────────────
    is_gap = smoothed <= max_ink * 0.06   # column has < 6 % of peak ink → gap

    segments, start = [], None
    for i in range(w):
        if not is_gap[i] and start is None:
            start = i
        elif is_gap[i] and start is not None:
            segments.append((start, i - 1))
            start = None
    if start is not None:
        segments.append((start, w - 1))

    if len(segments) > 1:
        result = []
        for x0, x1 in segments:
            sub = crop[:, x0:x1 + 1]
            if sub.shape[1] >= 5 and sub.max() > 30:
                result.append((sub, x_offset + x0))
        if len(result) > 1:
            return result

    # ── Phase 2: valley-based (touching / cursive letters) ────────────────
    inv_proj   = max_ink - smoothed
    min_dist   = int(est_char_w * 0.55)
    prominence = max_ink * 0.40     # deep threshold → only real char boundaries

    peaks, _ = find_peaks(inv_proj, distance=min_dist, prominence=prominence)
    if len(peaks) == 0:
        return [(crop, x_offset)]

    max_splits = max(1, int(w / est_char_w) - 1)
    if len(peaks) > max_splits:
        depths = inv_proj[peaks]
        keep   = np.argsort(depths)[-max_splits:]
        peaks  = sorted(peaks[keep].tolist())
    else:
        peaks = sorted(peaks.tolist())

    boundaries = [0] + peaks + [w]
    result = []
    for j in range(len(boundaries) - 1):
        x0, x1 = boundaries[j], boundaries[j + 1]
        sub = crop[:, x0:x1]
        if sub.shape[1] >= 5 and sub.max() > 30:
            result.append((sub, x_offset + x0))

    return result if len(result) > 1 else [(crop, x_offset)]


def segment_characters(img_pil, min_area=250, dilation_iters=5):
    """
    1. Invert the canvas (strokes → bright).
    2. Dilate 5 iterations — reconnects multi-stroke glyphs like 'v', 't', 'i',
       while clear spaces between letters survive as gaps.
    3. Label connected components; skip tiny noise blobs (min_area).
    4. Split any blob that is wider than ~1.6 characters using projection.
    Returns list of (crop_array, x_start) sorted left-to-right.
    """
    gray     = np.array(img_pil.convert('L'))
    inverted = 255 - gray
    binary   = (inverted > 30).astype(np.uint8)

    struct  = ndimage.generate_binary_structure(2, 2)
    dilated = ndimage.binary_dilation(binary, structure=struct,
                                      iterations=dilation_iters)
    labeled, num_features = ndimage.label(dilated)

    char_crops = []
    for label_id in range(1, num_features + 1):
        mask = labeled == label_id
        rows = np.where(np.any(mask, axis=1))[0]
        cols = np.where(np.any(mask, axis=0))[0]
        rmin, rmax = rows[0], rows[-1]
        cmin, cmax = cols[0], cols[-1]
        h = rmax - rmin + 1
        w = cmax - cmin + 1
        if h * w < min_area:
            continue                       # skip stray dots / noise
        crop = inverted[rmin:rmax + 1, cmin:cmax + 1]
        char_crops.extend(split_by_projection(crop, cmin, h))

    char_crops.sort(key=lambda x: x[1])
    return char_crops


def preprocess_char(char_array):
    """Centre a character crop in a square, resize to 28×28, normalise."""
    img  = Image.fromarray(char_array.astype(np.uint8))
    w, h = img.size
    pad  = max(w, h) // 4
    img  = ImageOps.expand(img, pad, fill=0)
    img  = img.resize((28, 28), Image.LANCZOS)
    tensor = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])(img)
    return tensor.unsqueeze(0).to(device), img


def run_prediction(img_pil):
    chars = segment_characters(img_pil)

    if not chars:
        print("No characters detected — draw something first.")
        return

    results = []
    for char_array, _ in chars:
        tensor, img28 = preprocess_char(char_array)
        with torch.no_grad():
            probs = torch.softmax(model(tensor), dim=1)[0]

        top5_p, top5_i = torch.topk(probs, 5)
        top5_p    = top5_p.cpu().numpy() * 100
        top5_c    = [classes[i] for i in top5_i.cpu()]
        confident = top5_p[0] >= CONF_THRESHOLD * 100

        results.append({
            'img28':     img28,
            'char':      top5_c[0] if confident else '?',
            'conf':      top5_p[0],
            'top5_c':    top5_c,
            'top5_p':    top5_p,
            'confident': confident,
        })

    word = ''.join(r['char'] for r in results)
    n    = len(results)

    fig, axes = plt.subplots(2, n, figsize=(max(4, 3.5 * n), 6),
                             gridspec_kw={'height_ratios': [1, 2]})
    if n == 1:
        axes = [[axes[0]], [axes[1]]]

    fig.suptitle(f'Predicted: "{word}"', fontsize=15, fontweight='bold', y=1.03)

    for i, r in enumerate(results):
        col = '#4CAF50' if r['confident'] else '#f44336'

        ax_img = axes[0][i]
        ax_img.imshow(np.array(r['img28']), cmap='gray', vmin=0, vmax=255)
        ax_img.set_title(f"'{r['char']}'  {r['conf']:.1f}%",
                         fontsize=10, color=col, fontweight='bold')
        ax_img.axis('off')

        ax_bar  = axes[1][i]
        labels  = [f"'{c}'" for c in r['top5_c'][::-1]]
        values  = r['top5_p'][::-1]
        bar_col = ([col] + ['#64B5F6'] * 4)[::-1]
        ax_bar.barh(labels, values, color=bar_col)
        ax_bar.set_xlim(0, 112)
        ax_bar.set_xlabel('Confidence (%)', fontsize=8)
        for j, v in enumerate(values):
            ax_bar.text(v + 0.5, j, f'{v:.1f}%', va='center', fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
CANVAS_SIZE = 280
canvas = Canvas(width=CANVAS_SIZE, height=CANVAS_SIZE)
canvas.sync_image_data = True

canvas.fill_style   = 'white'
canvas.fill_rect(0, 0, CANVAS_SIZE, CANVAS_SIZE)
canvas.stroke_style = 'black'
canvas.line_width   = 16
canvas.line_cap     = 'round'
canvas.line_join    = 'round'

_drawing = False
_last    = [0, 0]

def on_down(x, y):
    global _drawing
    _drawing = True
    _last[0], _last[1] = x, y

def on_move(x, y):
    if not _drawing: return
    with hold_canvas(canvas):
        canvas.begin_path()
        canvas.move_to(_last[0], _last[1])
        canvas.line_to(x, y)
        canvas.stroke()
    _last[0], _last[1] = x, y

def on_up(x, y):
    global _drawing
    _drawing = False

canvas.on_mouse_down(on_down)
canvas.on_mouse_move(on_move)
canvas.on_mouse_up(on_up)

clear_btn   = widgets.Button(description='Clear',   button_style='warning', icon='trash')
predict_btn = widgets.Button(description='Predict', button_style='success', icon='search')
status      = widgets.Label(value='Draw a character above, then click Predict.')
out         = widgets.Output()

def on_clear(b):
    with hold_canvas(canvas):
        canvas.clear()
        canvas.fill_style = 'white'
        canvas.fill_rect(0, 0, CANVAS_SIZE, CANVAS_SIZE)
    status.value = 'Canvas cleared.'
    with out: clear_output()

def on_predict(b):
    status.value = 'Running...'
    rgba    = canvas.get_image_data(0, 0, CANVAS_SIZE, CANVAS_SIZE)
    img_pil = Image.fromarray(rgba, 'RGBA').convert('RGB')
    with out:
        clear_output(wait=True)
        run_prediction(img_pil)
    status.value = 'Done! Draw another character and click Predict again.'

clear_btn.on_click(on_clear)
predict_btn.on_click(on_predict)

display(widgets.VBox([
    canvas,
    widgets.HBox([clear_btn, predict_btn]),
    status,
    out,
]))
